# Employee Attrition Prediction System
### AIDAS 2 — Mini Project

**Objective:** Explore employee data and build a machine-learning model to predict whether an employee is marked as having left (`Attrition = Yes`) or not (`Attrition = No`).

**Dataset file:** `employee_attrition_dataset.csv`  
**Features available:** Age, BusinessTravel, DailyRate, Department, DistanceFromHome, Education, EducationField  
**Target:** Attrition

The supplied CSV was cleaned for this project: `EmployeeCount` was removed because it is constant, and `EmployeeNumber` was removed because it is an identifier rather than a useful employee characteristic.

**Workflow:** Load data → data checks → EDA → preprocessing → compare models → evaluate → save model bundle for the Streamlit app.

## 1. Import libraries

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay, RocCurveDisplay,
    roc_auc_score
)

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 2. Load the cleaned dataset

In [ ]:
DATA_PATH = "employee_attrition_dataset.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Keep the CSV in the same folder as this notebook."
    )

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())

## 3. Data quality checks

In [ ]:
print("Column names:", df.columns.tolist())
print("\nData types:")
display(df.dtypes.to_frame("Data Type"))
print("\nMissing values:")
display(df.isna().sum().to_frame("Missing Values"))
print("\nDuplicate rows:", df.duplicated().sum())
print("\nTarget count:")
display(df["Attrition"].value_counts().rename_axis("Attrition").to_frame("Count"))
print("\nTarget percentage:")
display((df["Attrition"].value_counts(normalize=True) * 100).round(2).rename("Percentage").to_frame())

## 4. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Attrition")
plt.title("Employee Attrition Distribution")
plt.xlabel("Attrition")
plt.ylabel("Number of Employees")
plt.tight_layout()
plt.show()

print(f"Attrition rate: {100 * (df['Attrition'].eq('Yes').mean()):.2f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=df, x="Age", hue="Attrition", multiple="stack", bins=15, ax=axes[0])
axes[0].set_title("Age Distribution by Attrition")
sns.countplot(data=df, x="BusinessTravel", hue="Attrition", ax=axes[1])
axes[1].set_title("Attrition by Business Travel")
axes[1].tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.countplot(data=df, x="Department", hue="Attrition", ax=axes[0])
axes[0].set_title("Attrition by Department")
axes[0].tick_params(axis="x", rotation=15)
sns.boxplot(data=df, x="Attrition", y="DistanceFromHome", ax=axes[1])
axes[1].set_title("Distance from Home by Attrition")
plt.tight_layout()
plt.show()

In [ ]:
attrition_by_field = (
    df.assign(AttritionFlag=df["Attrition"].eq("Yes").astype(int))
      .groupby("EducationField")["AttritionFlag"].mean()
      .sort_values(ascending=False)
)
display((attrition_by_field * 100).round(2).rename("Attrition Rate (%)").to_frame())

plt.figure(figsize=(8, 4))
sns.barplot(x=attrition_by_field.index, y=attrition_by_field.values, errorbar=None)
plt.title("Attrition Rate by Education Field")
plt.ylabel("Attrition proportion")
plt.xlabel("Education Field")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

## 5. Prepare features and target

In [ ]:
data = df.dropna(subset=["Attrition"]).copy()
X = data.drop(columns=["Attrition"])
y = data["Attrition"].map({"No": 0, "Yes": 1})

if y.isna().any():
    raise ValueError("Target values must be 'Yes' or 'No'.")

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Features:", X.columns.tolist())
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("Target encoding: No = 0, Yes = 1")

## 6. Split data and create preprocessing pipeline

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

## 7. Train and compare models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=250, class_weight="balanced",
        min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1
    )
}

trained_models = {}
results = []

for model_name, classifier in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", classifier)
    ])
    pipeline.fit(X_train, y_train)

    pred = pipeline.predict(X_test)
    prob = pipeline.predict_proba(X_test)[:, 1]

    trained_models[model_name] = pipeline
    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision (Yes)": precision_score(y_test, pred, zero_division=0),
        "Recall (Yes)": recall_score(y_test, pred, zero_division=0),
        "F1-score (Yes)": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, prob)
    })

results_df = pd.DataFrame(results).sort_values("F1-score (Yes)", ascending=False)
display(results_df.round(3))

## 8. Evaluate the selected model

In [ ]:
best_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_name]
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("Selected model:", best_name)
print("\nClassification report:")
print(classification_report(
    y_test, y_pred, labels=[0, 1],
    target_names=["No Attrition", "Attrition"], zero_division=0
))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, labels=[0, 1],
    display_labels=["No", "Yes"], cmap="Blues"
)
plt.title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.show()

RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title(f"ROC Curve — {best_name}")
plt.tight_layout()
plt.show()

## 9. Feature importance

In [ ]:
classifier = best_model.named_steps["classifier"]
feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()

if hasattr(classifier, "feature_importances_"):
    importance_values = classifier.feature_importances_
elif hasattr(classifier, "coef_"):
    importance_values = np.abs(classifier.coef_[0])
else:
    importance_values = None

if importance_values is not None:
    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": importance_values
    }).sort_values("Importance", ascending=False)

    display(importance_df.head(15))
    plt.figure(figsize=(9, 5))
    sns.barplot(data=importance_df.head(15), x="Importance", y="Feature", errorbar=None)
    plt.title("Top Features Used by the Model")
    plt.tight_layout()
    plt.show()
else:
    print("Feature importance is not supported for this model.")

## 10. Save the trained model for Streamlit

In [ ]:
model_bundle = {
    "model": best_model,
    "feature_columns": X.columns.tolist(),
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "medians": X[numeric_features].median(numeric_only=True).to_dict(),
    "modes": {
        col: (X[col].mode(dropna=True).iloc[0] if not X[col].mode(dropna=True).empty else "")
        for col in categorical_features
    },
    "selected_model": best_name
}

joblib.dump(model_bundle, "employee_attrition_model.pkl")
print("Saved: employee_attrition_model.pkl")
print("Keep this file in the same folder as app.py.")

## 11. Conclusion

This project explores attrition patterns using the available employee features and compares two classification models. Because the dataset has limited columns, predictions are based only on age, business travel, daily rate, department, distance from home, education level, and education field.

**Responsible use:** This is an educational project using a public HR dataset. Model predictions may be wrong and should not be used as the sole basis for employment decisions.